In [0]:
SELECT *
FROM workspace.default.sample_superstore
LIMIT 10

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0.0,14.1694
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0.0,1.9656
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0.0,34.47


In [0]:
SELECT
    COUNT(DISTINCT `Order ID`)                        AS total_pedidos,
    COUNT(DISTINCT `Customer ID`)                     AS total_clientes,
    ROUND(SUM(Sales), 2)                              AS faturamento_total,
    ROUND(SUM(Profit), 2)                             AS lucro_total,
    ROUND(SUM(Sales) / COUNT(DISTINCT `Order ID`), 2) AS ticket_medio,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 2)          AS margem_lucro_pct
FROM workspace.default.sample_superstore

total_pedidos,total_clientes,faturamento_total,lucro_total,ticket_medio,margem_lucro_pct
5009,793,2297200.86,286397.02,458.61,12.47


In [0]:
SELECT
    Category,
    COUNT(DISTINCT `Order ID`)                        AS total_pedidos,
    ROUND(SUM(Sales), 2)                              AS faturamento,
    ROUND(SUM(Profit), 2)                             AS lucro,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 2)          AS margem_pct,
    ROUND(SUM(Sales) / COUNT(DISTINCT `Order ID`), 2) AS ticket_medio
FROM workspace.default.sample_superstore
GROUP BY Category
ORDER BY faturamento DESC

Category,total_pedidos,faturamento,lucro,margem_pct,ticket_medio
Technology,1544,836154.03,145454.95,17.4,541.55
Furniture,1764,741999.8,18451.27,2.49,420.63
Office Supplies,3742,719047.03,122490.8,17.04,192.16


Databricks visualization. Run in Databricks to view.

In [0]:

WITH mensal AS (
    SELECT
        DATE_FORMAT(`Order Date`, 'yyyy-MM')          AS ano_mes,
        ROUND(SUM(Sales), 2)                          AS faturamento,
        ROUND(SUM(Profit), 2)                         AS lucro
    FROM workspace.default.sample_superstore
    GROUP BY DATE_FORMAT(`Order Date`, 'yyyy-MM')
),
com_crescimento AS (
    SELECT
        ano_mes,
        faturamento,
        lucro,
        LAG(faturamento) OVER (ORDER BY ano_mes)      AS faturamento_mes_anterior,
        ROUND(
            (faturamento - LAG(faturamento) OVER (ORDER BY ano_mes))
            / LAG(faturamento) OVER (ORDER BY ano_mes) * 100
        , 2)                                          AS crescimento_pct
    FROM mensal
)
SELECT *
FROM com_crescimento
ORDER BY ano_mes

ano_mes,faturamento,lucro,faturamento_mes_anterior,crescimento_pct
2014-01,14236.89,2450.19,null,null
2014-02,4519.89,862.31,14236.89,-68.25
2014-03,55691.01,498.73,4519.89,1132.13
2014-04,28295.34,3488.84,55691.01,-49.19
2014-05,23648.29,2738.71,28295.34,-16.42
2014-06,34595.13,4976.52,23648.29,46.29
2014-07,33946.39,-841.48,34595.13,-1.88
2014-08,27909.47,5318.11,33946.39,-17.78
2014-09,81777.35,8328.1,27909.47,193.01
2014-10,31453.39,3448.26,81777.35,-61.54


Databricks visualization. Run in Databricks to view.

In [0]:
WITH subcategoria AS (
    SELECT
        Category,
        `Sub-Category`,
        ROUND(SUM(Sales), 2)                        AS faturamento,
        ROUND(SUM(Profit), 2)                       AS lucro,
        ROUND(SUM(Profit) / SUM(Sales) * 100, 2)    AS margem_pct,
        COUNT(DISTINCT `Order ID`)                  AS total_pedidos
    FROM workspace.default.sample_superstore
    GROUP BY Category, `Sub-Category`
),
com_ranking AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY Category
            ORDER BY faturamento DESC
        )                                           AS rank_faturamento_categoria
    FROM subcategoria
)
SELECT *
FROM com_ranking
ORDER BY Category, rank_faturamento_categoria

Category,Sub-Category,faturamento,lucro,margem_pct,total_pedidos,rank_faturamento_categoria
Furniture,Chairs,328449.1,26590.17,8.1,576,1
Furniture,Tables,206965.53,-17725.48,-8.56,307,2
Furniture,Bookcases,114880.0,-3472.56,-3.02,224,3
Furniture,Furnishings,91705.16,13059.14,14.24,877,4
Office Supplies,Storage,223843.61,21278.83,9.51,777,1
Office Supplies,Binders,203412.73,30221.76,14.86,1316,2
Office Supplies,Appliances,107532.16,18138.01,16.87,451,3
Office Supplies,Paper,78479.21,34053.57,43.39,1191,4
Office Supplies,Supplies,46673.54,-1189.1,-2.55,187,5
Office Supplies,Art,27118.79,6527.79,24.07,731,6


In [0]:

WITH regional AS (
    SELECT
        Region,
        ROUND(SUM(Sales), 2)                            AS faturamento,
        ROUND(SUM(Profit), 2)                           AS lucro,
        ROUND(SUM(Profit) / SUM(Sales) * 100, 2)        AS margem_pct,
        COUNT(DISTINCT `Customer ID`)                   AS total_clientes,
        COUNT(DISTINCT `Order ID`)                      AS total_pedidos
    FROM workspace.default.sample_superstore
    GROUP BY Region
),
com_participacao AS (
    SELECT
        *,
        ROUND(faturamento / SUM(faturamento) OVER () * 100, 2) AS participacao_pct
    FROM regional
)
SELECT *
FROM com_participacao
ORDER BY faturamento DESC

Region,faturamento,lucro,margem_pct,total_clientes,total_pedidos,participacao_pct
West,725457.82,108418.45,14.94,686,1611,31.58
East,678781.24,91522.78,13.48,674,1401,29.55
Central,501239.89,39706.36,7.92,629,1175,21.82
South,391721.91,46749.43,11.93,512,822,17.05


Databricks visualization. Run in Databricks to view.

In [0]:

WITH faixas AS (
    SELECT
        CASE
            WHEN Discount = 0            THEN '01 - Sem desconto'
            WHEN Discount BETWEEN 0.01 AND 0.10 THEN '02 - Desconto 1-10%'
            WHEN Discount BETWEEN 0.11 AND 0.20 THEN '03 - Desconto 11-20%'
            WHEN Discount BETWEEN 0.21 AND 0.30 THEN '04 - Desconto 21-30%'
            WHEN Discount > 0.30         THEN '05 - Desconto acima 30%'
        END                                             AS faixa_desconto,
        Sales,
        Profit
    FROM workspace.default.sample_superstore
),
resumo AS (
    SELECT
        faixa_desconto,
        COUNT(*)                                        AS total_itens,
        ROUND(SUM(Sales), 2)                            AS faturamento,
        ROUND(SUM(Profit), 2)                           AS lucro,
        ROUND(SUM(Profit) / SUM(Sales) * 100, 2)        AS margem_pct
    FROM faixas
    GROUP BY faixa_desconto
)
SELECT
    *,
    ROUND(SUM(faturamento) OVER () , 2)                 AS faturamento_total,
    ROUND(faturamento / SUM(faturamento) OVER () * 100, 2) AS participacao_pct
FROM resumo
ORDER BY faixa_desconto

faixa_desconto,total_itens,faturamento,lucro,margem_pct,faturamento_total,participacao_pct
01 - Sem desconto,4798,1087908.47,320987.6,29.51,2297200.85,47.36
02 - Desconto 1-10%,94,54369.35,9029.18,16.61,2297200.85,2.37
03 - Desconto 11-20%,3709,792152.89,91756.3,11.58,2297200.85,34.48
04 - Desconto 21-30%,227,103226.65,-10369.28,-10.05,2297200.85,4.49
05 - Desconto acima 30%,1166,259543.49,-125006.78,-48.16,2297200.85,11.3


Databricks visualization. Run in Databricks to view.